# Dataset preparation
Discover the binary dataset, create or reuse the persistent 70/30 stratified split, and inspect the preprocessing pipelines. Re-running this notebook reuses existing CSV files rather than reshuffling examples.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

from applied_ai_midterm.config import load_config
from applied_ai_midterm.data import discover_images, prepare_splits
from applied_ai_midterm.transforms import (
    SRGANPairTransform,
    classifier_transform,
    denormalize_imagenet,
    denormalize_srgan,
)

config = load_config(Path("configs/config.yaml"))
raw_data_dir = Path("data/raw")
splits_dir = Path("data/splits")

In [ ]:
discovered = discover_images(raw_data_dir)
display(discovered.groupby(["class_name", "label"]).size().rename("image_count").to_frame())

In [ ]:
splits = prepare_splits(
    raw_data_dir,
    splits_dir,
    train_ratio=config.train_ratio,
    random_seed=config.random_seed,
)
split_counts = pd.concat(
    [
        splits.train.assign(split="train"),
        splits.test.assign(split="test"),
    ]
).groupby(["split", "class_name", "label"]).size().rename("image_count")
display(split_counts.to_frame())

In [ ]:
sample_rows = splits.train.groupby("class_name", group_keys=False).head(2)
figure, axes = plt.subplots(1, len(sample_rows), figsize=(4 * len(sample_rows), 4))
for axis, row in zip(axes, sample_rows.itertuples(), strict=True):
    with Image.open(row.filepath) as image:
        axis.imshow(image.convert("RGB"))
    axis.set_title(f"Original: {row.class_name}")
    axis.axis("off")
figure.tight_layout()

In [ ]:
classifier_train = classifier_transform(training=True, image_size=config.high_resolution_size)
figure, axes = plt.subplots(1, len(sample_rows), figsize=(4 * len(sample_rows), 4))
for axis, row in zip(axes, sample_rows.itertuples(), strict=True):
    with Image.open(row.filepath) as image:
        transformed = classifier_train(image.convert("RGB"))
    axis.imshow(denormalize_imagenet(transformed).permute(1, 2, 0))
    axis.set_title(f"Classifier transform: {row.class_name}")
    axis.axis("off")
figure.tight_layout()

In [ ]:
pair_transform = SRGANPairTransform(
    training=True,
    low_resolution_size=config.low_resolution_size,
    high_resolution_size=config.high_resolution_size,
)
figure, axes = plt.subplots(len(sample_rows), 2, figsize=(7, 3 * len(sample_rows)))
for row_axes, row in zip(axes, sample_rows.itertuples(), strict=True):
    with Image.open(row.filepath) as image:
        low_resolution, high_resolution = pair_transform(image.convert("RGB"))
    row_axes[0].imshow(denormalize_srgan(low_resolution).permute(1, 2, 0))
    row_axes[0].set_title(f"32×32 low resolution: {row.class_name}")
    row_axes[1].imshow(denormalize_srgan(high_resolution).permute(1, 2, 0))
    row_axes[1].set_title(f"128×128 target: {row.class_name}")
    for axis in row_axes:
        axis.axis("off")
figure.tight_layout()